In [ ]:
!pip install streamlit pyngrok bcrypt pyjwt

KeyboardInterrupt: 

In [ ]:
%%writefile app.py
import streamlit as st
import sqlite3
import bcrypt
import jwt
import datetime
import random
import smtplib
import re
from email.mime.text import MIMEText

SECRET_KEY = "textmorph_secret_key"

EMAIL_ID = "abcyas2810@gmail.com"
EMAIL_PASSWORD = "vplh zyaf ajak ylpq"

ADMIN_EMAIL = "admin@gmail.com"
ADMIN_PASSWORD = "admin123"

#  DATABASE

conn = sqlite3.connect("users.db", check_same_thread=False)
c = conn.cursor()

c.execute("""
CREATE TABLE IF NOT EXISTS users(
username TEXT,
email TEXT PRIMARY KEY,
password BLOB,
question TEXT,
answer TEXT,
failed_attempts INTEGER DEFAULT 0,
lock_until TEXT,
old_password BLOB
)
""")

conn.commit()

# JWT

def create_token(email):

    payload = {
        "user": email,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)
    }

    return jwt.encode(payload, SECRET_KEY, algorithm="HS256")

# OTP

def send_otp(email):

    otp = str(random.randint(100000,999999))

    msg = MIMEText(f"Your OTP is {otp}")
    msg["Subject"] = "TextMorph OTP Verification"
    msg["From"] = EMAIL_ID
    msg["To"] = email

    server = smtplib.SMTP("smtp.gmail.com",587)
    server.starttls()
    server.login(EMAIL_ID,EMAIL_PASSWORD)
    server.send_message(msg)
    server.quit()

    return otp

#UI

st.set_page_config(page_title="TextMorph Authentication")

menu = ["Signup","Login","Forgot Password"]
choice = st.sidebar.selectbox("Menu",menu)

#  SIGNUP

if choice == "Signup":

    st.title("Signup")

    username = st.text_input("Username")
    email = st.text_input("Email")

    password = st.text_input("Password",type="password")
    confirm = st.text_input("Confirm Password",type="password")

    question = st.selectbox("Security Question",[
        "What is your pet name?",
        "What is your mother's maiden name?",
        "What is your favorite teacher?"
    ])

    answer = st.text_input("Answer")

    if st.button("Send OTP"):

        st.session_state["otp"] = send_otp(email)
        st.success("OTP sent to email")

    user_otp = st.text_input("Enter OTP")

    if st.button("Signup"):

        if not all([username,email,password,confirm,answer]):
            st.error("All fields required")

        elif password != confirm:
            st.error("Passwords do not match")

        elif not re.match(r'^(?=.*[A-Za-z])(?=.*\d).{6,}$', password):
            st.error("Password must contain at least one letter and one number")

        elif user_otp != st.session_state.get("otp"):
            st.error("Invalid OTP")

        else:

            hashed = bcrypt.hashpw(password.encode(),bcrypt.gensalt())

            c.execute("INSERT INTO users VALUES(?,?,?,?,?,?,?,?)",
            (username,email,hashed,question,answer,0,None,None))

            conn.commit()

            st.success("Signup Successful")

# LOGIN

elif choice == "Login":

    st.title("Login")

    email = st.text_input("Email")
    password = st.text_input("Password",type="password")

    if st.button("Login"):

        # Admin login

        if email == ADMIN_EMAIL and password == ADMIN_PASSWORD:

            st.session_state["admin"] = True
            st.success("Admin Login Successful")
            st.stop()

        c.execute("SELECT * FROM users WHERE email=?",(email,))
        user = c.fetchone()

        if not user:

            st.error("User not found")
            st.stop()

        failed_attempts = user[5]
        lock_until = user[6]

        if lock_until:

            lock_time = datetime.datetime.fromisoformat(lock_until)

            if datetime.datetime.now() < lock_time:

                st.error("Account locked for 5 minutes")
                st.stop()

        if bcrypt.checkpw(password.encode(),user[2]):

            token = create_token(email)

            st.session_state["user"] = user[0]
            st.session_state["token"] = token

            c.execute("UPDATE users SET failed_attempts=0 WHERE email=?",(email,))
            conn.commit()

            st.success("Login Successful")
            st.code(token)

        else:

            failed_attempts += 1

            if failed_attempts >= 3:

                lock_time = datetime.datetime.now() + datetime.timedelta(minutes=5)

                c.execute("UPDATE users SET failed_attempts=?, lock_until=? WHERE email=?",
                (failed_attempts,lock_time.isoformat(),email))

            else:

                c.execute("UPDATE users SET failed_attempts=? WHERE email=?",
                (failed_attempts,email))

            conn.commit()

            st.error("Wrong Password")

# DASHBOARD

if "user" in st.session_state:

    st.title("User Dashboard")

    st.success(f"Welcome {st.session_state['user']}")

    if st.button("Logout"):

        st.session_state.clear()
        st.rerun()

# ADMIN DASHBOARD

if "admin" in st.session_state:

    st.title("Admin Dashboard")

    c.execute("SELECT username,email FROM users")
    users = c.fetchall()

    st.table(users)

    if st.button("Logout Admin"):

        st.session_state.clear()
        st.rerun()

# FORGOT PASSWORD

elif choice == "Forgot Password":

    st.title("Reset Password")

    email = st.text_input("Enter Email")

    if st.button("Verify"):

        c.execute("SELECT * FROM users WHERE email=?",(email,))
        user = c.fetchone()

        if user:

            st.session_state["reset_user"] = user

        else:

            st.error("Email not found")

    if "reset_user" in st.session_state:

        user = st.session_state["reset_user"]

        st.info(user[3])

        ans = st.text_input("Answer")

        new_pass = st.text_input("New Password",type="password")

        if st.button("Reset"):

            if ans == user[4]:

                if user[7] and bcrypt.checkpw(new_pass.encode(),user[7]):

                    st.error("Cannot reuse old password")

                else:

                    hashed = bcrypt.hashpw(new_pass.encode(),bcrypt.gensalt())

                    c.execute("UPDATE users SET old_password=?, password=? WHERE email=?",
                    (user[2],hashed,email))

                    conn.commit()

                    st.success("Password Updated")

            else:

                st.error("Wrong Answer")

Overwriting app.py


In [ ]:
!ngrok config add-authtoken 3AZsHso7WKL7FjfBGVC4OhA7ruW_4sfjvShFrQrV41Y8Qny5q

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!streamlit run app.py --server.port 8501 &




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.189.177.107:8501

/content/app.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)
/usr/local/lib/python3.12/dist-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/content/app.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)


In [ ]:
from pyngrok import ngrok

ngrok.kill()

url = ngrok.connect(8501)
print("Public URL:", url)

Public URL: NgrokTunnel: "https://denunciatorily-bearlike-ricki.ngrok-free.dev" -> "http://localhost:8501"
